# AI Agent on Databricks

This notebook walks through building an AI agent on Databricks using the Mosaic AI Agent Framework. It covers:
1. Setting up the environment
2. Connecting to a Databricks-hosted LLM
3. Making basic LLM calls
4. Building a tool-calling agent

## Environment Inspection

First, we inspect the currently installed packages to understand what's already available in the Databricks cluster environment.

In [0]:
%pip list

## Install Required Packages

We upgrade and install the three core packages needed for this notebook:
- **`mlflow`** – for experiment tracking and auto-logging of LLM calls
- **`databricks-openai`** – provides the `UCFunctionToolkit` for wrapping Unity Catalog functions as agent tools
- **`databricks-agents`** – the Mosaic AI Agent Framework SDK for building and deploying agents on Databricks

In [0]:
%pip install -U -qqqq mlflow databricks-openai databricks-agents

In [0]:
dbutils.library.restartPython()

## Verify Package Installation

Confirm that `databricks-agents` was installed correctly and check its version. This notebook was developed against **v1.9.3**.

In [0]:
%pip show databricks-agents

In [0]:
import warnings
warnings.filterwarnings('ignore')

## Select an Available LLM Endpoint

Databricks hosts LLMs as Model Serving endpoints. This cell auto-discovers which endpoint is available in the current workspace by probing two candidates in order:
1. `databricks-claude-3-7-sonnet` (Anthropic Claude)
2. `databricks-meta-llama-3-3-70b-instruct` (Meta LLaMA)

The first endpoint that responds successfully is used for the rest of the notebook. An assertion ensures we fail fast if neither is reachable.

In [0]:
LLM_ENDPOINT_NAME = None

from databricks.sdk import WorkspaceClient
def is_endpoint_available(endpoint_name):
  try:
    client = WorkspaceClient().serving_endpoints.get_open_ai_client()
    client.chat.completions.create(model=endpoint_name, messages=[{"role": "user", "content": "What is AI?"}])
    return True
  except Exception:
    return False

client = WorkspaceClient()
for candidate_endpoint_name in ["databricks-claude-3-7-sonnet", "databricks-meta-llama-3-3-70b-instruct"]:
    if is_endpoint_available(candidate_endpoint_name):
        LLM_ENDPOINT_NAME = candidate_endpoint_name
assert LLM_ENDPOINT_NAME is not None, "Please specify LLM_ENDPOINT_NAME" 
    

In [0]:
LLM_ENDPOINT_NAME

In [0]:
import json
import mlflow
from databricks.sdk import WorkspaceClient

mlflow.openai.autolog()
openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

prompt = "What is the capital of Nepal ?"

openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
    )

## Basic LLM Call with MLflow Auto-logging

Here we make our first direct call to the LLM endpoint using an OpenAI-compatible client obtained from the Databricks `WorkspaceClient`. 

`mlflow.openai.autolog()` is enabled so that every LLM call is automatically traced and logged to the MLflow experiment — no manual instrumentation needed. The response object is a standard OpenAI `ChatCompletion`.

In [0]:
response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
    )

In [0]:
response

In [0]:
response.id

In [0]:
response.choices[0].message.content

## Wrapping the LLM Call in a Helper Function

We refactor the raw API call into a reusable `run_llm(prompt)` helper that:
- Sends a user prompt to the selected LLM endpoint
- Returns the assistant's response as a list of message dictionaries (role + content)

This makes it easy to call the LLM consistently throughout the notebook.

In [0]:
def run_llm(prompt):
    result_msgs = []
    response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
    )
    msg = response.choices[0].message
    result_msgs.append(msg.to_dict())
    return result_msgs

In [0]:
resp = run_llm("What is the capital of Nepal ?")
for message in resp:
    print(f'{message["role"]}: {message["content"]}')

## Building a Tool-Calling Agent

Now we extend the simple LLM call into a proper **agent** that can use tools.

### Key components:
- **`UCFunctionToolkit`** – wraps Unity Catalog (UC) functions as OpenAI-compatible tool schemas. Here we expose `system.ai.python_exec`, which lets the LLM execute arbitrary Python code inside a sandboxed Databricks environment.
- **`DatabricksFunctionClient`** – handles actually invoking UC functions when the LLM decides to call a tool.
- **`call_tool()`** – routes the LLM's tool call request to the correct UC function executor.
- **`run_agent()`** – the main agent loop: sends the prompt + tool definitions to the LLM, checks if a tool call was requested, executes it, and returns the full message history (assistant turn + tool result).

> **Note:** This is a single-step agent loop. A production agent would continue iterating — feeding tool results back to the LLM — until the model returns a final answer with no further tool calls.

In [0]:
from databricks_openai import UCFunctionToolkit, DatabricksFunctionClient
from databricks.sdk import WorkspaceClient

mlflow.openai.autolog()
openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

client = DatabricksFunctionClient()
builtin_tools = UCFunctionToolkit(
    function_names=["system.ai.python_exec"], client=client
).tools

def call_tool(tool_name, parameters):
    if tool_name == "system__ai__python_exec":
        return DatabricksFunctionClient().execute_function(
            "system.ai.python_exec", parameters=parameters
        )
    raise ValueError(f"Unknown tool: {tool_name}")


def run_agent(prompt):
    result_msgs = []
    response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
        tools=builtin_tools,
    )
    msg = response.choices[0].message
    result_msgs.append(msg.to_dict())

    if msg.tool_calls:
        call = msg.tool_calls[0]
        tool_result = call_tool(call.function.name, json.loads(call.function.arguments))
        result_msgs.append(
            {
                "role": "tool",
                "content": tool_result.value,
                "name": call.function.name,
                "tool_call_id": call.id,
            }
        )
    return result_msgs

In [0]:
answer = run_agent("What is the best place to visit in Nepal ?")

In [0]:
for message in answer:
    print(f'{message["role"]}: {message["content"]}')

## Testing the Agent with a Computation Task

This test demonstrates tool use in action. When asked to compute the square root of 9, the LLM recognizes this is a calculation and calls `system.ai.python_exec` to run Python code rather than guessing the answer. The tool returns `3.0`, which is the actual computed result.

This illustrates the key difference between a plain LLM call (which would answer from training data) and an agent (which can take actions and get grounded results).

In [0]:
import json
answer = run_agent("What is the square root of 9?")
for message in answer:
    print(f'{message["role"]}: {message["content"]}')